# LeetCode 3010 — Divide an Array Into Subarrays With Minimum Cost I bằng Python

Link đề: https://leetcode.com/problems/divide-an-array-into-subarrays-with-minimum-cost-i/ · Độ khó: **Easy** · Chủ đề: Array, Sorting, Enumeration

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumCost(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với vét cạn), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia mảng thành **đúng 3 mảng con liên tiếp, không giao nhau, phủ hết mảng**; **chi phí** của một mảng là **phần tử đầu tiên** của nó; tìm cách chia để **tổng chi phí nhỏ nhất**.
2. Nhận ra cấu trúc của mọi cách chia: ba chi phí là `nums[0]` (đầu mảng 1, luôn cố định) và `nums[i]`, `nums[j]` với `0 < i < j` (đầu mảng 2 và mảng 3).
3. Hiểu vì sao bài rút về: **`nums[0]` cộng hai số nhỏ nhất trong `nums[1:]`** (hai số ở **hai vị trí khác nhau**).
4. Đi từ cách **vét cạn mọi cặp điểm cắt** → **sắp xếp phần còn lại, lấy hai số đầu** → **một lượt duy trì hai giá trị nhỏ nhất** → **`heapq.nsmallest`**.
5. Tránh các lỗi kinh điển: chọn **ba số nhỏ nhất của cả mảng** (bỏ qua việc `nums[0]` bắt buộc), dùng **cùng một phần tử hai lần**, vô tình **sắp xếp luôn mảng đầu vào** rồi dùng `nums[0]`, và cập nhật hai giá trị nhỏ nhất sai thứ tự.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài `n`. **Chi phí** của một mảng là **giá trị phần tử đầu tiên** của nó. Ví dụ chi phí của `[1, 2, 3]` là `1`, của `[3, 4, 1]` là `3`.

Bạn cần **chia `nums` thành `3` mảng con liên tiếp, không rỗng, không giao nhau** (mọi phần tử thuộc đúng một mảng con). Hãy trả về **tổng chi phí nhỏ nhất** có thể đạt được của ba mảng con đó.

**Ví dụ 1**

- Input: `nums = [1, 2, 3, 12]`
- Output: `6`
- Giải thích: cách tốt nhất là `[1]`, `[2]`, `[3, 12]` với tổng chi phí `1 + 2 + 3 = 6`. Các cách chia khác (`[1], [2, 3], [12]` có chi phí `1 + 2 + 12 = 15`; `[1, 2], [3], [12]` có chi phí `1 + 3 + 12 = 16`) đều tệ hơn.

**Ví dụ 2**

- Input: `nums = [5, 4, 3]`
- Output: `12`
- Giải thích: chỉ có một cách chia `[5], [4], [3]` với tổng chi phí `5 + 4 + 3 = 12`.

**Ví dụ 3**

- Input: `nums = [10, 3, 1, 1]`
- Output: `12`
- Giải thích: cách tốt nhất là `[10, 3], [1], [1]` với tổng chi phí `10 + 1 + 1 = 12`.

**Ràng buộc:** `3 <= n <= 50`; `1 <= nums[i] <= 50`.

In [1]:
from typing import List
import heapq
import random

examples = [
    ([1, 2, 3, 12], 6),
    ([5, 4, 3], 12),
    ([10, 3, 1, 1], 12),
    ([1, 1, 1], 3),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 3, 12] -> kỳ vọng 6
nums = [5, 4, 3] -> kỳ vọng 12
nums = [10, 3, 1, 1] -> kỳ vọng 12
nums = [1, 1, 1] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Mảng con liên tiếp và "điểm cắt"

Chia `nums` thành `3` mảng con liên tiếp tương đương chọn **hai điểm cắt** `i < j` với `1 <= i < j <= n - 1`: mảng 1 là `nums[0:i]`, mảng 2 là `nums[i:j]`, mảng 3 là `nums[j:n]`. Ràng buộc `1 <= i` và `j <= n - 1` đảm bảo cả ba mảng **không rỗng**.

In [2]:
nums = [1, 2, 3, 12]
n = len(nums)
for i in range(1, n - 1):
    for j in range(i + 1, n):
        print((i, j), nums[0:i], nums[i:j], nums[j:n])

(1, 2) [1] [2] [3, 12]
(1, 3) [1] [2, 3] [12]
(2, 3) [1, 2] [3] [12]


### 3.2. Chi phí của một mảng là phần tử đầu

Với điểm cắt `(i, j)`, ba chi phí là `nums[0]` (đầu mảng 1), `nums[i]` (đầu mảng 2), `nums[j]` (đầu mảng 3). Tổng: `nums[0] + nums[i] + nums[j]`. Vì chi phí của mảng 1 **luôn** là `nums[0]`, nó **bắt buộc** nằm trong tổng.

In [3]:
nums = [10, 3, 1, 1]
i, j = 2, 3
parts = [nums[0:i], nums[i:j], nums[j:]]
print(parts, [p[0] for p in parts], sum(p[0] for p in parts))    # [[10, 3], [1], [1]] [10, 1, 1] 12

[[10, 3], [1], [1]] [10, 1, 1] 12


### 3.3. Cắt lát `nums[1:]`

`nums[1:]` là mảng bỏ phần tử đầu tiên (`nums[0]`). Ta sẽ chọn hai số nhỏ nhất **từ phần này**. Cắt lát tạo bản sao nên không làm đổi `nums`.

In [4]:
nums = [10, 3, 1, 1]
rest = nums[1:]
print(rest, nums)                          # [3, 1, 1] [10, 3, 1, 1]

[3, 1, 1] [10, 3, 1, 1]


### 3.4. Sắp xếp và lấy hai phần tử đầu

`sorted(rest)` cho danh sách tăng dần; `[:2]` lấy hai phần tử đầu (hai số nhỏ nhất, kể cả khi bằng nhau vì mỗi phần tử ở **một vị trí riêng**). `sorted` không đổi `rest`; `rest.sort()` thì đổi.

In [5]:
rest = [3, 1, 1]
print(sorted(rest), sorted(rest)[:2], sum(sorted(rest)[:2]))      # [1, 1, 3] [1, 1] 2

[1, 1, 3] [1, 1] 2


### 3.5. Hai giá trị nhỏ nhất trong một lượt

Giữ hai biến `m1 <= m2` (hai nhỏ nhất đến giờ), khởi tạo bằng một giá trị rất lớn (`float("inf")`). Với mỗi `x`: nếu `x < m1` thì **đẩy `m1` xuống làm `m2`** rồi `m1 = x`; ngược lại nếu `x < m2` thì `m2 = x`. Nhớ thứ tự `m2 = m1` **trước** `m1 = x`.

In [6]:
INF = float("inf")
m1 = m2 = INF
for x in [3, 1, 1, 7]:
    if x < m1:
        m2, m1 = m1, x                     # m1 cũ xuống làm m2, x làm m1
    elif x < m2:
        m2 = x
    print(f"x={x}: m1={m1} m2={m2}")

x=3: m1=3 m2=inf
x=1: m1=1 m2=3
x=1: m1=1 m2=1
x=7: m1=1 m2=1


### 3.6. `heapq.nsmallest(k, iterable)`

`heapq.nsmallest(2, a)` trả về danh sách `k` phần tử nhỏ nhất theo thứ tự tăng dần, hiệu quả hơn sắp xếp toàn bộ khi `k` nhỏ (`O(n log k)`).

In [7]:
print(heapq.nsmallest(2, [3, 1, 1, 7]))     # [1, 1]
print(heapq.nsmallest(1, [3, 1, 1, 7]))     # [1]

[1, 1]
[1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [10, 3, 1, 1]`, liệt kê mọi cách chia thành `3` mảng (`n = 4` nên có `C(3, 2) = 3` cách):

| điểm cắt `(i, j)` | ba mảng | chi phí `nums[0] + nums[i] + nums[j]` |
|---|---|---|
| `(1, 2)` | `[10], [3], [1, 1]` | `10 + 3 + 1 = 14` |
| `(1, 3)` | `[10], [3, 1], [1]` | `10 + 3 + 1 = 14` |
| `(2, 3)` | `[10, 3], [1], [1]` | `10 + 1 + 1 = 12` |

Nhỏ nhất là `12`. Quan sát: tổng luôn là **`nums[0]`** cộng **hai phần tử ở hai chỉ số khác nhau thuộc `1..n-1`**; và **mọi cặp chỉ số `i < j` trong `1..n-1` đều là một cách chia hợp lệ**. Vậy muốn tổng nhỏ nhất ta chọn **hai số nhỏ nhất của `nums[1:]`**: `1` và `1`, tổng `10 + 1 + 1 = 12`.

In [8]:
nums = [10, 3, 1, 1]
n = len(nums)
for i in range(1, n - 1):
    for j in range(i + 1, n):
        print(f"cắt tại ({i}, {j}): {nums[0:i]} {nums[i:j]} {nums[j:]} -> chi phí {nums[0]} + {nums[i]} + {nums[j]} = {nums[0] + nums[i] + nums[j]}")

cắt tại (1, 2): [10] [3] [1, 1] -> chi phí 10 + 3 + 1 = 14
cắt tại (1, 3): [10] [3, 1] [1] -> chi phí 10 + 3 + 1 = 14
cắt tại (2, 3): [10, 3] [1] [1] -> chi phí 10 + 1 + 1 = 12


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp điểm cắt

**Ý tưởng.** Hai vòng lặp `i` từ `1` đến `n - 2`, `j` từ `i + 1` đến `n - 1`; tính `nums[0] + nums[i] + nums[j]` và giữ giá trị nhỏ nhất.

In [9]:
def minimum_cost_brute(nums: List[int]) -> int:
    n = len(nums)
    best = float("inf")
    for i in range(1, n - 1):                          # điểm cắt thứ nhất: đầu mảng 2
        for j in range(i + 1, n):                      # điểm cắt thứ hai: đầu mảng 3
            best = min(best, nums[0] + nums[i] + nums[j])
    return best


for nums, expected in examples:
    print(nums, "->", minimum_cost_brute(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 12] -> 6 (kỳ vọng 6 )
[5, 4, 3] -> 12 (kỳ vọng 12 )
[10, 3, 1, 1] -> 12 (kỳ vọng 12 )
[1, 1, 1] -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Có `O(n²)` cặp `(i, j)`, mỗi cặp `O(1)`: **Time `O(n²)`**, **Space `O(1)`**.

**Nút thắt.** Với `n <= 50` rất nhanh, nhưng nhìn vào bảng ở mục 4 ta thấy **không cần thử từng cặp**: tổng chỉ phụ thuộc vào **hai số nhỏ nhất** của `nums[1:]`.

## 6. Ý tưởng thuật toán / bước đột phá

Với `0 < i < j <= n - 1`, tổng là `nums[0] + nums[i] + nums[j]`. Khi `i`, `j` chạy qua **mọi** cặp chỉ số khác nhau của `1..n-1`, `{nums[i], nums[j]}` là **mọi** cặp phần tử (ở hai vị trí khác nhau) của `nums[1:]`. Tổng nhỏ nhất đạt khi `nums[i]` và `nums[j]` là **hai phần tử nhỏ nhất** của `nums[1:]`.

Không cần chọn `i < j` đúng thứ tự giá trị: nếu hai số nhỏ nhất nằm ở chỉ số `p < q`, đặt `i = p`, `j = q`. Vậy:

> **đáp án = `nums[0]` + (số nhỏ nhất của `nums[1:]`) + (số nhỏ thứ hai của `nums[1:]`)**

(Số nhỏ thứ hai tính **theo vị trí**: nếu giá trị nhỏ nhất xuất hiện hai lần thì cả hai đều được lấy, như ví dụ 3.)

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp `nums[1:]`, lấy hai phần tử đầu

**Giải thích.** `rest = sorted(nums[1:])`, đáp án `nums[0] + rest[0] + rest[1]`. `sorted` tạo danh sách mới nên không đụng đến `nums`.

In [10]:
def minimum_cost_sort(nums: List[int]) -> int:
    rest = sorted(nums[1:])                            # bỏ nums[0], sắp xếp phần còn lại (bản sao)
    return nums[0] + rest[0] + rest[1]                 # nums[0] + hai số nhỏ nhất


for nums, expected in examples:
    print(nums, "->", minimum_cost_sort(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 12] -> 6 (kỳ vọng 6 )
[5, 4, 3] -> 12 (kỳ vọng 12 )
[10, 3, 1, 1] -> 12 (kỳ vọng 12 )
[1, 1, 1] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** **Time `O(n log n)`**, **Space `O(n)`**.

### Cách 3 — Một lượt, giữ hai giá trị nhỏ nhất `m1`, `m2`

**Giải thích.** Duyệt `nums[1:]` (hoặc `range(1, n)`), duy trì `m1 <= m2` như mục 3.5. Không sắp xếp, không bộ nhớ phụ.

In [11]:
def minimum_cost_two_min(nums: List[int]) -> int:
    m1 = m2 = float("inf")                             # hai giá trị nhỏ nhất của nums[1:]
    for k in range(1, len(nums)):
        x = nums[k]
        if x < m1:
            m2, m1 = m1, x                             # m1 cũ trở thành m2
        elif x < m2:
            m2 = x
    return nums[0] + m1 + m2


for nums, expected in examples:
    print(nums, "->", minimum_cost_two_min(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 12] -> 6 (kỳ vọng 6 )
[5, 4, 3] -> 12 (kỳ vọng 12 )
[10, 3, 1, 1] -> 12 (kỳ vọng 12 )
[1, 1, 1] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — `heapq.nsmallest(2, nums[1:])`

**Giải thích.** Dùng sẵn hàm của thư viện: `a, b = heapq.nsmallest(2, nums[1:])`, đáp án `nums[0] + a + b`. Gọn và vẫn gần `O(n)` (với `k = 2`).

In [12]:
def minimum_cost_heap(nums: List[int]) -> int:
    a, b = heapq.nsmallest(2, nums[1:])                # hai số nhỏ nhất, theo vị trí
    return nums[0] + a + b


for nums, expected in examples:
    print(nums, "->", minimum_cost_heap(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 12] -> 6 (kỳ vọng 6 )
[5, 4, 3] -> 12 (kỳ vọng 12 )
[10, 3, 1, 1] -> 12 (kỳ vọng 12 )
[1, 1, 1] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 4:** **Time `O(n)`** (`k = 2` cố định: `O(n log 2)`), **Space `O(1)`** ngoài bản sao `nums[1:]` (hoặc `O(n)` nếu tính bản sao).

In [13]:
# Code nộp LeetCode cho Cách 3 (O(n) thời gian, O(1) bộ nhớ)
from typing import List

class Solution:
    def minimumCost(self, nums: List[int]) -> int:
        m1 = m2 = float("inf")
        for k in range(1, len(nums)):
            x = nums[k]
            if x < m1:
                m2, m1 = m1, x
            elif x < m2:
                m2 = x
        return nums[0] + m1 + m2

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 với `[10, 7, 3, 9, 1, 4]` và (b) Cách 1 chỉ ra điểm cắt nào cho kết quả nhỏ nhất.

In [14]:
def trace_two_min(nums):
    m1 = m2 = float("inf")
    print("nums =", nums, "| nums[0] =", nums[0], "| xét nums[1:] =", nums[1:])
    for k in range(1, len(nums)):
        x = nums[k]
        if x < m1:
            action = f"x < m1 -> m2={m1}, m1={x}"
            m2, m1 = m1, x
        elif x < m2:
            action = f"m1 <= x < m2 -> m2={x}"
            m2 = x
        else:
            action = "x >= m2 -> bỏ qua"
        print(f"k={k} x={x:>2}: {action:<28} | m1={m1} m2={m2}")
    total = nums[0] + m1 + m2
    print(f"=> {nums[0]} + {m1} + {m2} = {total}")
    return total


assert trace_two_min([10, 7, 3, 9, 1, 4]) == 14
assert trace_two_min([10, 3, 1, 1]) == 12

nums = [10, 7, 3, 9, 1, 4] | nums[0] = 10 | xét nums[1:] = [7, 3, 9, 1, 4]
k=1 x= 7: x < m1 -> m2=inf, m1=7       | m1=7 m2=inf
k=2 x= 3: x < m1 -> m2=7, m1=3         | m1=3 m2=7
k=3 x= 9: x >= m2 -> bỏ qua            | m1=3 m2=7
k=4 x= 1: x < m1 -> m2=3, m1=1         | m1=1 m2=3
k=5 x= 4: x >= m2 -> bỏ qua            | m1=1 m2=3
=> 10 + 1 + 3 = 14
nums = [10, 3, 1, 1] | nums[0] = 10 | xét nums[1:] = [3, 1, 1]
k=1 x= 3: x < m1 -> m2=inf, m1=3       | m1=3 m2=inf
k=2 x= 1: x < m1 -> m2=3, m1=1         | m1=1 m2=3
k=3 x= 1: m1 <= x < m2 -> m2=1         | m1=1 m2=1
=> 10 + 1 + 1 = 12


In [15]:
def trace_brute(nums):
    n = len(nums)
    best, best_cut = float("inf"), None
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            c = nums[0] + nums[i] + nums[j]
            mark = ""
            if c < best:
                best, best_cut = c, (i, j)
                mark = "  <- tốt nhất mới"
            print(f"cắt ({i},{j}): {nums[0]}+{nums[i]}+{nums[j]}={c}{mark}")
    print("=> tốt nhất:", best, "tại", best_cut)
    return best


assert trace_brute([10, 7, 3, 9, 1, 4]) == 14

cắt (1,2): 10+7+3=20  <- tốt nhất mới
cắt (1,3): 10+7+9=26
cắt (1,4): 10+7+1=18  <- tốt nhất mới
cắt (1,5): 10+7+4=21
cắt (2,3): 10+3+9=22
cắt (2,4): 10+3+1=14  <- tốt nhất mới
cắt (2,5): 10+3+4=17
cắt (3,4): 10+9+1=20
cắt (3,5): 10+9+4=23
cắt (4,5): 10+1+4=15
=> tốt nhất: 14 tại (2, 4)


Bảng tóm tắt (Cách 3, `[10, 7, 3, 9, 1, 4]`):

| `k` | `x` | `m1` | `m2` |
|---|---|---|---|
| (đầu) | | `inf` | `inf` |
| 1 | 7 | 7 | `inf` |
| 2 | 3 | 3 | 7 |
| 3 | 9 | 3 | 7 |
| 4 | 1 | 1 | 3 |
| 5 | 4 | 1 | 3 |

Đáp án `10 + 1 + 3 = 14`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (một lượt, hai giá trị nhỏ nhất). Đúng class `Solution`, đúng method `minimumCost(self, nums: List[int]) -> int`.

In [16]:
from typing import List

class Solution:
    def minimumCost(self, nums: List[int]) -> int:
        m1 = m2 = float("inf")                       # hai giá trị nhỏ nhất của nums[1:]
        for k in range(1, len(nums)):                # nums[0] luôn là chi phí của mảng 1 nên bỏ qua
            x = nums[k]
            if x < m1:
                m2, m1 = m1, x                       # m1 cũ trở thành m2, x là m1 mới
            elif x < m2:
                m2 = x
        return nums[0] + m1 + m2


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.minimumCost(nums))

[1, 2, 3, 12] -> 6
[5, 4, 3] -> 12
[10, 3, 1, 1] -> 12
[1, 1, 1] -> 3


## 10. Vì sao thuật toán đúng

- **Cấu trúc của mọi cách chia.** Chia thành ba mảng liên tiếp tương ứng với hai điểm cắt `0 < i < j <= n - 1`. Chi phí là `nums[0] + nums[i] + nums[j]` (mỗi mảng lấy phần tử đầu). Chi phí mảng 1 luôn là `nums[0]`.
- **Mọi cặp vị trí đều dùng được.** Với **bất kỳ** hai chỉ số khác nhau `p < q` trong `1..n-1`, chọn `i = p`, `j = q` cho một cách chia hợp lệ (ba mảng không rỗng: `nums[0:p]` có `p >= 1` phần tử, `nums[p:q]` có `q - p >= 1`, `nums[q:]` có `n - q >= 1`). Nên tập các tổng đạt được là `nums[0] + nums[p] + nums[q]` trên mọi cặp `p < q`.
- **Cực tiểu.** Tổng nhỏ nhất của `nums[p] + nums[q]` trên mọi cặp vị trí khác nhau là **tổng của hai phần tử nhỏ nhất theo vị trí** của `nums[1:]`: đây là hai số đầu sau khi sắp xếp (nếu giá trị trùng nhau thì lấy cả hai bản, vì chúng ở hai vị trí khác nhau).
- **Cách 3 giữ đúng hai nhỏ nhất.** Bất biến: sau mỗi phần tử, `m1 <= m2` là hai giá trị nhỏ nhất (theo vị trí) của các phần tử đã xét. Phần tử mới `x < m1` làm `m1` cũ thành `m2`; `m1 <= x < m2` thay `m2`; `x >= m2` không ảnh hưởng.
- **Đủ phần tử.** `n >= 3` nên `nums[1:]` có ít nhất hai phần tử, `m1` và `m2` đều là số thật (không còn `inf`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (vét cạn điểm cắt):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (sắp xếp `nums[1:]`):** Time `O(n log n)`, Space `O(n)`.
- **Cách 3 (một lượt `m1`, `m2`):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (`heapq.nsmallest(2, ...)`):** Time `O(n)`, Space `O(n)` cho bản sao (hoặc `O(1)` nếu duyệt chỉ số).
- Phải đọc mọi phần tử của `nums[1:]` nên `O(n)` là tối ưu. Ở đây `n <= 50`.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "brute": minimum_cost_brute,
    "sort": minimum_cost_sort,
    "two_min": minimum_cost_two_min,
    "heap": minimum_cost_heap,
    "submit": sol.minimumCost,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 1, 1], 3, "n = 3 tối thiểu, toàn bằng nhau")
check_all([7, 2, 9], 18, "n = 3: chỉ một cách chia, tổng = 7 + 2 + 9")
check_all([50, 50, 50], 150, "n = 3, giá trị lớn nhất")
check_all([1, 50, 50, 50], 101, "phần tử đầu nhỏ, phần còn lại lớn")
check_all([50, 1, 1, 50], 52, "phần tử đầu lớn, hai số 1 ở giữa")
check_all([50, 1, 1], 52, "n = 3 phần tử đầu lớn")
check_all([3, 5, 4, 1, 2], 6, "hai số nhỏ nhất 1 và 2 ở cuối: 3 + 1 + 2")
check_all([3, 1, 2, 5, 4], 6, "hai số nhỏ nhất ở đầu của phần còn lại: 3 + 1 + 2")
check_all([5, 4, 4, 4, 4], 13, "nhiều số bằng nhau nhỏ nhất: 5 + 4 + 4")
check_all([1] * 50, 3, "n = 50 toàn 1")
check_all(list(range(50, 0, -1)), 50 + 1 + 2, "n = 50 giảm dần 50..1: 50 + 1 + 2")

PASS n = 3 tối thiểu, toàn bằng nhau -> 3
PASS n = 3: chỉ một cách chia, tổng = 7 + 2 + 9 -> 18
PASS n = 3, giá trị lớn nhất -> 150
PASS phần tử đầu nhỏ, phần còn lại lớn -> 101
PASS phần tử đầu lớn, hai số 1 ở giữa -> 52
PASS n = 3 phần tử đầu lớn -> 52
PASS hai số nhỏ nhất 1 và 2 ở cuối: 3 + 1 + 2 -> 6
PASS hai số nhỏ nhất ở đầu của phần còn lại: 3 + 1 + 2 -> 6
PASS nhiều số bằng nhau nhỏ nhất: 5 + 4 + 4 -> 13
PASS n = 50 toàn 1 -> 3
PASS n = 50 giảm dần 50..1: 50 + 1 + 2 -> 53


### 12.2. Tính chất và phản ví dụ

In [19]:
# Tính chất 1: kết quả >= nums[0] + 2 (hai chi phí còn lại >= 1 mỗi cái)
random.seed(11)
for _ in range(500):
    nums = [random.randint(1, 50) for _ in range(random.randint(3, 50))]
    assert sol.minimumCost(nums) >= nums[0] + 2
print("PASS kết quả >= nums[0] + 2")

# Tính chất 2: kết quả <= nums[0] + nums[1] + nums[2] (một cách chia cụ thể)
for _ in range(500):
    nums = [random.randint(1, 50) for _ in range(random.randint(3, 50))]
    assert sol.minimumCost(nums) <= nums[0] + nums[1] + nums[2]
print("PASS kết quả <= nums[0] + nums[1] + nums[2]")

# Tính chất 3: xáo trộn nums[1:] không đổi kết quả
for _ in range(300):
    nums = [random.randint(1, 50) for _ in range(random.randint(3, 30))]
    rest = nums[1:]
    random.shuffle(rest)
    assert sol.minimumCost(nums) == sol.minimumCost([nums[0]] + rest)
print("PASS thứ tự trong nums[1:] không quan trọng")

# Tính chất 4: thay đổi nums[0] tăng d thì kết quả tăng đúng d
for _ in range(300):
    nums = [random.randint(1, 30) for _ in range(random.randint(3, 20))]
    d = random.randint(1, 10)
    bigger = [nums[0] + d] + nums[1:]
    assert sol.minimumCost(bigger) == sol.minimumCost(nums) + d
print("PASS nums[0] cộng thẳng vào kết quả")

# Tính chất 5: đầu vào không bị sửa
nums = [10, 3, 1, 1]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: ba số nhỏ nhất của cả mảng
def sai_ba_nho_nhat(nums):
    return sum(sorted(nums)[:3])

assert sai_ba_nho_nhat([10, 3, 1, 1]) == 5 != 12
print("PASS phản ví dụ: ba số nhỏ nhất của cả mảng cho", sai_ba_nho_nhat([10, 3, 1, 1]), "thay vì 12")

# Phản ví dụ 2: dùng cùng một phần tử nhỏ nhất hai lần
def sai_hai_lan(nums):
    return nums[0] + 2 * min(nums[1:])

assert sai_hai_lan([10, 3, 1, 4]) == 12 and sol.minimumCost([10, 3, 1, 4]) == 14
print("PASS phản ví dụ: dùng min hai lần cho", sai_hai_lan([10, 3, 1, 4]), "thay vì 14 ở [10, 3, 1, 4]")

PASS kết quả >= nums[0] + 2
PASS kết quả <= nums[0] + nums[1] + nums[2]
PASS thứ tự trong nums[1:] không quan trọng
PASS nums[0] cộng thẳng vào kết quả
PASS đầu vào không bị sửa
PASS phản ví dụ: ba số nhỏ nhất của cả mảng cho 5 thay vì 12
PASS phản ví dụ: dùng min hai lần cho 12 thay vì 14 ở [10, 3, 1, 4]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(3010)
for trial in range(3000):
    n = random.randint(3, 50)
    hi = random.choice([2, 5, 20, 50])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = minimum_cost_brute(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp vét cạn")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp vét cạn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chọn ba số nhỏ nhất của cả mảng

In [21]:
def sai_ba_nho(nums):
    return sum(sorted(nums)[:3])


print("sai  :", sai_ba_nho([10, 3, 1, 1]), "| đúng:", Solution().minimumCost([10, 3, 1, 1]))

sai  : 5 | đúng: 12


**Vì sao sai:** chi phí của mảng thứ nhất **luôn** là `nums[0]` (phần tử đầu của mảng gốc, nên `10` ở ví dụ này), dù nó lớn đến đâu. Chỉ hai chi phí còn lại mới "được chọn". Phải lấy `nums[0]` cộng hai số nhỏ nhất của `nums[1:]`.

### Lỗi 2: Dùng cùng một phần tử hai lần

In [22]:
def sai_min_hai_lan(nums):
    return nums[0] + 2 * min(nums[1:])


print("sai  :", sai_min_hai_lan([10, 3, 1, 4]), "| đúng:", Solution().minimumCost([10, 3, 1, 4]))

sai  : 12 | đúng: 14


**Vì sao sai:** mỗi mảng con bắt đầu tại **một vị trí riêng**, nên hai chi phí phải đến từ **hai vị trí khác nhau**. `min(nums[1:])` chỉ ứng với một vị trí; nhân đôi nó giả sử có hai số `1` trong khi mảng chỉ có một. Cần số nhỏ nhất **và** số nhỏ thứ hai theo vị trí.

### Lỗi 3: Sắp xếp luôn mảng đầu vào rồi dùng `nums[0]`

In [23]:
def sai_sort_tai_cho(nums):
    nums.sort()                                          # sắp xếp ngay trên mảng gốc: nums[0] đã là số nhỏ nhất
    return nums[0] + nums[1] + nums[2]


data = [10, 3, 1, 1]
print("sai  :", sai_sort_tai_cho(data), "| đúng:", Solution().minimumCost([10, 3, 1, 1]), "| mảng sau khi gọi:", data)

sai  : 5 | đúng: 12 | mảng sau khi gọi: [1, 1, 3, 10]


**Vì sao sai:** sau `nums.sort()` thì `nums[0]` không còn là phần tử đầu **ban đầu** (`10`) mà là số nhỏ nhất, đồng thời mảng của người gọi bị đổi thứ tự. Hãy lấy `first = nums[0]` **trước**, hoặc dùng `sorted(nums[1:])` (bản sao).

### Lỗi 4: Cập nhật hai giá trị nhỏ nhất sai thứ tự

In [24]:
def sai_cap_nhat(nums):
    m1 = m2 = float("inf")
    for x in nums[1:]:
        if x < m1:
            m1 = x                                       # quên đẩy m1 cũ xuống m2
        elif x < m2:
            m2 = x
    return nums[0] + m1 + m2


print("sai  :", sai_cap_nhat([10, 5, 4, 3]), "| đúng:", Solution().minimumCost([10, 5, 4, 3]))

sai  : inf | đúng: 17


**Vì sao sai:** khi gặp số nhỏ hơn `m1`, giá trị `m1` cũ phải trở thành `m2` (nó vẫn là số nhỏ thứ hai). Bỏ bước `m2 = m1` làm `m2` không được cập nhật nếu dãy giảm dần (`5, 4, 3`), kết quả là `inf`. Dùng `m2, m1 = m1, x`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "chia mảng thành `k` mảng liên tiếp, chi phí = **phần tử đầu**" → chi phí của mảng 1 là `nums[0]` cố định; các mảng còn lại tự do chọn **bất kỳ vị trí nào** làm điểm đầu.
- Công thức nhớ: **`nums[0]` + tổng `k - 1` số nhỏ nhất của `nums[1:]`**.
- Thấy "chọn hai/ba giá trị nhỏ nhất" → `sorted(...)[:k]`, `heapq.nsmallest(k, ...)` hoặc duy trì `m1`, `m2` trong một lượt.
- Hai giá trị bằng nhau ở hai vị trí khác nhau vẫn là **hai** phần tử (không loại trùng).
- Khởi tạo `float("inf")` cho "nhỏ nhất đến giờ". Nhớ đẩy `m1` cũ xuống `m2`.
- Luôn kiểm tra `n` nhỏ nhất (`n = 3`: chỉ một cách chia) và vét cạn để đối chiếu.

## 15. Biến thể của bài

1. Chia thành **`k`** mảng con (tổng quát): `nums[0]` cộng `k - 1` số nhỏ nhất của `nums[1:]`.
2. Chi phí là **phần tử cuối** của mảng thay vì phần tử đầu.
3. Chi phí là **phần tử lớn nhất** của mảng (bài khác hẳn, cần quy hoạch động).
4. Trả về **một cách chia** đạt chi phí nhỏ nhất (các điểm cắt).
5. Đếm **số cách chia** đạt chi phí nhỏ nhất.

Thử biến thể 1, 2 và 4:

In [25]:
def minimum_cost_k(nums, k):
    return nums[0] + sum(heapq.nsmallest(k - 1, nums[1:]))


def minimum_cost_last_element(nums):
    # chi phí = phần tử cuối: mảng cuối luôn kết thúc ở nums[-1]; hai mảng đầu kết thúc ở hai vị trí khác nhau trong 0..n-2
    return nums[-1] + sum(heapq.nsmallest(2, nums[:-1]))


def best_cut(nums):
    n = len(nums)
    best, cut = float("inf"), None
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            c = nums[0] + nums[i] + nums[j]
            if c < best:
                best, cut = c, (i, j)
    return best, cut


print("1. k = 3:", minimum_cost_k([10, 3, 1, 1], 3), "| k = 2:", minimum_cost_k([10, 3, 1, 1], 2))   # 12, 11
print("2. chi phí = phần tử cuối:", minimum_cost_last_element([4, 1, 7, 2, 9]))                     # 9 + 1 + 2 = 12
print("4. cách chia tốt nhất:", best_cut([10, 3, 1, 1]))                                             # (12, (2, 3))
assert minimum_cost_k([10, 3, 1, 1], 3) == Solution().minimumCost([10, 3, 1, 1])
assert minimum_cost_k([10, 3, 1, 1], 2) == 11
assert minimum_cost_last_element([4, 1, 7, 2, 9]) == 12
assert best_cut([10, 3, 1, 1]) == (12, (2, 3))

1. k = 3: 12 | k = 2: 11
2. chi phí = phần tử cuối: 12
4. cách chia tốt nhất: (12, (2, 3))


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp điểm cắt | `nums[0] + nums[i] + nums[j]` | `O(n²)` | `O(1)` | hiểu đề, đáp án chuẩn |
| 2. Sắp xếp `nums[1:]` | lấy hai số đầu | `O(n log n)` | `O(n)` | ngắn gọn |
| 3. Một lượt `m1`, `m2` | duy trì hai giá trị nhỏ nhất | `O(n)` | `O(1)` | **cách nộp bài**, tối ưu |
| 4. `heapq.nsmallest(2, ...)` | thư viện | `O(n)` | `O(n)` | code gọn, dễ mở rộng sang `k` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[7, 3, 5][1:]`, `sorted([4, 1, 3])[:2]`, `heapq.nsmallest(2, [5, 2, 9, 2])`, `Solution().minimumCost([8, 1, 2])`, `Solution().minimumCost([2, 9, 9, 1, 5])`.

**Bài 2 (tính tay).** Với `nums = [6, 4, 2, 8, 3]`: liệt kê **mọi** cặp điểm cắt `(i, j)` (có `C(4, 2) = 6` cặp), tính chi phí từng cách và cho biết nhỏ nhất.

**Bài 3 (điền TODO).** Hoàn thành `minimum_cost_todo(nums)` theo Cách 2.

In [26]:
def minimum_cost_todo(nums):
    # TODO 1: rest = nums[1:] đã sắp xếp tăng dần (dùng sorted, không dùng nums.sort())
    # TODO 2: trả về nums[0] + rest[0] + rest[1]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_cost_two_min2(nums)` theo **một lượt** (Cách 3) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `minimum_cost_k2(nums, k)`: chia thành `k` mảng (`3 <= k <= n`), chi phí là phần tử đầu. (b) Viết `count_optimal_splits(nums)`: **số cặp điểm cắt** `(i, j)` cho tổng chi phí nhỏ nhất. (c) Viết `maximum_cost(nums)`: **tổng chi phí lớn nhất** (ba mảng, chi phí là phần tử đầu).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[3, 5]`, `[1, 3]`, `[2, 2]`, `11` (`8 + 1 + 2`), `8` (`nums[0] = 2`, `nums[1:] = [9, 9, 1, 5]` có hai số nhỏ nhất `1` và `5`: `2 + 1 + 5`). Bài 3: `rest = sorted(nums[1:])`; `return nums[0] + rest[0] + rest[1]`.
- **Mức 2 (vừa):** Bài 2: `nums = [6, 4, 2, 8, 3]`: các `(i, j)` và `nums[i] + nums[j]`: `(1,2)`: `4+2`, `(1,3)`: `4+8`, `(1,4)`: `4+3`, `(2,3)`: `2+8`, `(2,4)`: `2+3`, `(3,4)`: `8+3`. Nhỏ nhất `2 + 3 = 5` nên tổng `6 + 5 = 11`. Bài 4: xem Cách 3.
- **Mức 3 (sâu):** Bài 5(a): `nums[0] + sum(heapq.nsmallest(k - 1, nums[1:]))`. 5(b): với mỗi cặp tính chi phí và đếm cặp bằng giá trị nhỏ nhất. 5(c): `nums[0] + hai số lớn nhất của nums[1:]` (dùng `heapq.nlargest(2, ...)`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert [7, 3, 5][1:] == [3, 5] and sorted([4, 1, 3])[:2] == [1, 3]
assert heapq.nsmallest(2, [5, 2, 9, 2]) == [2, 2]
assert Solution().minimumCost([8, 1, 2]) == 11
assert Solution().minimumCost([2, 9, 9, 1, 5]) == 8

# Bài 2
nums2 = [6, 4, 2, 8, 3]
costs = {(i, j): nums2[0] + nums2[i] + nums2[j] for i in range(1, 5) for j in range(i + 1, 5)}
assert len(costs) == 6 and min(costs.values()) == 11
assert Solution().minimumCost(nums2) == 11

# Bài 3
def minimum_cost_todo_ans(nums):
    rest = sorted(nums[1:])
    return nums[0] + rest[0] + rest[1]

for nums, expected in examples:
    assert minimum_cost_todo_ans(nums) == expected

# Bài 4
def minimum_cost_two_min2(nums):
    m1 = m2 = float("inf")
    for x in nums[1:]:
        if x < m1:
            m2, m1 = m1, x
        elif x < m2:
            m2 = x
    return nums[0] + m1 + m2

for nums, expected in examples:
    assert minimum_cost_two_min2(nums) == expected

# Bài 5(a)
def minimum_cost_k2(nums, k):
    return nums[0] + sum(sorted(nums[1:])[:k - 1])

assert minimum_cost_k2([10, 3, 1, 1], 3) == 12
assert minimum_cost_k2([10, 3, 1, 1], 4) == 15

# Bài 5(b)
def count_optimal_splits(nums):
    n = len(nums)
    costs = [nums[0] + nums[i] + nums[j] for i in range(1, n - 1) for j in range(i + 1, n)]
    return costs.count(min(costs))

assert count_optimal_splits([10, 3, 1, 1]) == 1
assert count_optimal_splits([5, 1, 1, 1]) == 3
assert count_optimal_splits([1, 2, 3, 12]) == 1

# Bài 5(c)
def maximum_cost(nums):
    return nums[0] + sum(heapq.nlargest(2, nums[1:]))

assert maximum_cost([10, 3, 1, 1]) == 10 + 3 + 1
assert maximum_cost([1, 2, 3, 12]) == 1 + 12 + 3
for _ in range(100):
    arr = [random.randint(1, 20) for _ in range(random.randint(3, 12))]
    n = len(arr)
    brute_max = max(arr[0] + arr[i] + arr[j] for i in range(1, n - 1) for j in range(i + 1, n))
    assert maximum_cost(arr) == brute_max
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`nums[0]` cộng hai số nhỏ nhất của `nums[1:]`)

```python
from typing import List

class Solution:
    def minimumCost(self, nums: List[int]) -> int:
        rest = sorted(nums[1:])
        return nums[0] + rest[0] + rest[1]
```

**Checklist**

- [ ] `nums[0]` luôn nằm trong tổng (chi phí của mảng thứ nhất)?
- [ ] Hai số còn lại lấy từ **`nums[1:]`**, ở **hai vị trí khác nhau** (không dùng một phần tử hai lần, không loại trùng giá trị)?
- [ ] Không sắp xếp tại chỗ `nums.sort()` rồi dùng `nums[0]` (đã bị đổi)?
- [ ] Nếu một lượt: `m2, m1 = m1, x` (đẩy `m1` cũ xuống `m2`) và `elif x < m2: m2 = x`?
- [ ] Đã thử: `n = 3` (một cách chia), hai số nhỏ nhất bằng nhau, giá trị nhỏ nhất ở đầu/cuối, `n = 50`?
- [ ] Skeleton đúng `class Solution` / `def minimumCost(self, nums: List[int]) -> int`?